## How Stable Diffusion Connects to the VAE You Built

You built a VAE in `02_autoencoders.ipynb`. Stable Diffusion **uses a VAE internally** —
this is why it is called a *Latent* Diffusion Model (Rombach et al., CVPR 2022).

```
Text prompt ──► CLIP text encoder ──► Text embeddings
                                              │
Image (512×512) ──► VAE Encoder ──► Latent (64×64×4)
                                              │
                                     U-Net Denoiser  ◄── runs T denoising steps here
                                              │
                                     Denoised Latent
                                              │
                                     VAE Decoder ──► Generated Image (512×512)
```

The denoising happens entirely in the **compressed latent space** — approximately 8× smaller
than pixel space. This is what makes diffusion practical on consumer hardware.

**Why this architecture matters:**
- The **VAE bottleneck** = the same idea as in `02_autoencoders.ipynb` (compress, then decode)
- The **diffusion** = the new part: iterative denoising *inside* that latent representation
- Newer models (SD3, FLUX.1) replace the U-Net with a **Diffusion Transformer (DiT)** —
  applying the same attention mechanism from Transformers directly to the denoising process

**Paper:** Rombach et al. "High-Resolution Image Synthesis with Latent Diffusion Models"
CVPR 2022 — [arxiv.org/abs/2112.10752](https://arxiv.org/abs/2112.10752)

# Diffusion Models for Image Generation

**Module 03 | Notebook 3 of 3**

## Introduction

In this notebook, we'll explore **diffusion models**, the current state-of-the-art approach for image generation. Diffusion models power tools like Stable Diffusion, DALL-E, and Midjourney, and have become the industry standard in 2026.

### What You'll Learn

- How diffusion models work (forward and reverse diffusion)
- Text-to-image generation with Stable Diffusion
- Prompt engineering for image generation
- Comparison: VAEs vs Diffusion Models

### Prerequisites

- Understanding of neural networks
- Familiarity with image data
- Python and PyTorch basics

## 1. Understanding Diffusion Models

Diffusion models work through two processes:

1. **Forward Diffusion**: Gradually add noise to an image until it becomes pure noise
2. **Reverse Diffusion**: Learn to remove noise step-by-step to generate images

### Key Concepts

| Concept | Description |
|---------|-------------|
| Noise Schedule | Controls how noise is added/removed |
| Denoising Steps | Number of iterations to generate an image |
| Latent Space | Compressed representation for efficiency |
| Guidance Scale | How closely to follow the text prompt |

## 1b. The Forward Process — Visualized

Before using a production pipeline, let's build the core intuition with plain numpy (no GPU needed).

The **forward diffusion process** adds a little Gaussian noise at each of T timesteps.
After enough steps, the original image becomes indistinguishable from pure noise.

The model learns to **reverse** this process: given a noisy image at step t, predict
what it looked like at step t-1. Repeating this from pure noise (step T) back to step 0
generates a new image.

```
Original ──► Slightly noisy ──► More noisy ──► ... ──► Pure noise
  x₀              x₁              x₂                      xT

          ◄─── Model learns to reverse each arrow ────
```

### Classifier-Free Guidance (how `guidance_scale` works)

The denoising model runs **twice** at each step — once conditioned on the text prompt,
once without any conditioning (unconditional). The final prediction is their interpolation:

```
prediction = uncond_output + guidance_scale × (cond_output − uncond_output)
```

- `guidance_scale = 1.0` → pure unconditional (ignores the prompt entirely)
- `guidance_scale = 7.5` → strong prompt following (typical default)
- `guidance_scale > 15` → over-saturated, prone to artifacts

This is why `guidance_scale=1` looks generic and `guidance_scale=15` can look over-cooked.

In [ ]:
# Forward Diffusion Visualization — pure numpy, no GPU needed
import numpy as np
import matplotlib.pyplot as plt

# Use a simple grayscale gradient image as our "photo"
# (avoids needing to load MNIST just for this demo)
np.random.seed(42)
original = np.zeros((28, 28), dtype=np.float32)
original[8:20, 8:20] = 1.0   # White square = our "signal"

def add_noise_at_step(image, t, T=100, beta_start=1e-4, beta_end=0.02):
    """
    Add Gaussian noise corresponding to diffusion timestep t.
    
    Linear noise schedule: beta increases from beta_start to beta_end.
    noisy_image = sqrt(alpha_bar_t) * original + sqrt(1 - alpha_bar_t) * noise
    
    alpha_bar_t = product of (1 - beta_s) for s = 0..t
    As t increases, alpha_bar_t → 0, so the signal fades and noise dominates.
    """
    betas = np.linspace(beta_start, beta_end, T)
    alphas = 1.0 - betas
    alpha_bars = np.cumprod(alphas)  # cumulative product

    alpha_bar_t = alpha_bars[min(t, T-1)]
    noise = np.random.randn(*image.shape)
    noisy = np.sqrt(alpha_bar_t) * image + np.sqrt(1.0 - alpha_bar_t) * noise
    return np.clip(noisy, 0, 1)

# Visualize the forward process at different timesteps
timesteps = [0, 10, 25, 50, 75, 99]
T = 100

fig, axes = plt.subplots(1, len(timesteps), figsize=(14, 2.5))
for ax, t in zip(axes, timesteps):
    noisy = add_noise_at_step(original, t, T=T)
    ax.imshow(noisy, cmap='gray', vmin=0, vmax=1)
    ax.set_title(f't = {t}', fontsize=11)
    ax.axis('off')

plt.suptitle('Forward Diffusion: Clean Image → Pure Noise (linear schedule, T=100)', fontsize=12)
plt.tight_layout()
plt.show()

print("At t=0: clean signal. At t=99: indistinguishable from random noise.")
print("Stable Diffusion runs this in LATENT SPACE (via a VAE) — much more efficient than pixels.")
print("The U-Net learns to reverse each arrow — predicting the denoised image at t-1 from t.")

## 2. Setting Up Stable Diffusion

We'll use the `diffusers` library from Hugging Face, which provides easy access to state-of-the-art diffusion models.

In [ ]:
# Install required libraries
!pip install -q diffusers transformers accelerate torch torchvision

In [ ]:
import torch
from diffusers import StableDiffusionPipeline
from PIL import Image
import matplotlib.pyplot as plt

# Check if GPU is available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

## 3. Loading the Model

We'll use **Stable Diffusion v1.5** from Stability AI — a widely-used open-source
text-to-image model. It's a good learning reference because:

- Its architecture (VAE encoder + U-Net denoiser + CLIP text encoder) is well-documented
- It runs on both CPU (slowly) and GPU (fast)
- The concepts transfer directly to more modern models (SDXL, SD3, FLUX)

> **Note on modern models:** By 2025, the field has moved to SDXL (2023), SD3 / FLUX.1 (2024),
> and video models (2024-2025). After completing this notebook, explore
> `stabilityai/stable-diffusion-xl-base-1.0` for higher quality, or
> `black-forest-labs/FLUX.1-schnell` for the current state of the art.

In [ ]:
# Load Stable Diffusion pipeline
model_id = "stabilityai/stable-diffusion-v1-5"  # Public model, no access restrictions

# Use float16 for faster inference if GPU is available
if device == "cuda":
    pipe = StableDiffusionPipeline.from_pretrained(
        model_id,
        torch_dtype=torch.float16
    )
else:
    pipe = StableDiffusionPipeline.from_pretrained(model_id)

pipe = pipe.to(device)
print("Model loaded successfully!")

## 4. Text-to-Image Generation

Let's generate our first image from a text prompt!

In [ ]:
# Simple prompt
prompt = "a beautiful sunset over mountains, digital art"

# Generate image
image = pipe(prompt).images[0]

# Display
plt.figure(figsize=(8, 8))
plt.imshow(image)
plt.axis('off')
plt.title(f"Prompt: {prompt}")
plt.show()

## 5. Prompt Engineering

The quality of generated images heavily depends on the prompt. Here are some best practices:

### Prompt Structure
```
[Subject] + [Style] + [Details] + [Quality modifiers]
```

### Examples of Good Prompts
- "a majestic lion, oil painting, golden hour lighting, highly detailed, 4k"
- "futuristic cityscape, cyberpunk style, neon lights, rainy night, cinematic"
- "cute robot character, pixar style, 3d render, soft lighting, trending on artstation"

In [ ]:
# Try different prompts
prompts = [
    "a cozy coffee shop, warm lighting, watercolor painting",
    "astronaut riding a horse on mars, photorealistic, 8k",
    "abstract geometric patterns, vibrant colors, modern art"
]

# Generate and display
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, prompt in enumerate(prompts):
    image = pipe(prompt, num_inference_steps=30).images[0]
    axes[idx].imshow(image)
    axes[idx].axis('off')
    axes[idx].set_title(prompt[:40] + "...", fontsize=10)

plt.tight_layout()
plt.show()

## 6. Advanced Parameters

Let's explore key parameters that control image generation:

- **num_inference_steps**: More steps = higher quality but slower (default: 50)
- **guidance_scale**: How closely to follow the prompt (default: 7.5)
- **negative_prompt**: What to avoid in the image
- **seed**: For reproducible results

In [ ]:
prompt = "a serene japanese garden, cherry blossoms, koi pond, traditional architecture"
negative_prompt = "blurry, low quality, distorted, ugly"

# Generate with different guidance scales
guidance_scales = [3, 7.5, 15]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, scale in enumerate(guidance_scales):
    image = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        guidance_scale=scale,
        num_inference_steps=30,
        generator=torch.manual_seed(42)  # Same seed for comparison
    ).images[0]
    
    axes[idx].imshow(image)
    axes[idx].axis('off')
    axes[idx].set_title(f"Guidance Scale: {scale}")

plt.tight_layout()
plt.show()

## 7. Comparison: VAEs vs Diffusion Models

Let's compare the two generative approaches we've learned:

| Aspect | VAEs | Diffusion Models |
|--------|------|------------------|
| **Training** | Encoder + Decoder | Denoising network |
| **Generation Speed** | Fast (single pass) | Slow (iterative) |
| **Image Quality** | Good | Excellent |
| **Diversity** | Moderate | High |
| **Control** | Limited | High (via prompts) |
| **Use Cases** | Compression, latent space | Image generation, editing |

### When to Use Each

**Use VAEs when:**
- You need fast generation
- You want to work with latent representations
- You need compression

**Use Diffusion Models when:**
- Image quality is paramount
- You need text-to-image generation
- You want fine-grained control via prompts

## 8. Practical Exercise

**Task**: Create a series of images for a children's book

Requirements:
1. Generate 3 images with a consistent style
2. Use appropriate prompts for children's illustrations
3. Experiment with different parameters to achieve the best results

In [ ]:
# Your solution here
# Example prompts:
# - "a friendly dragon reading a book, children's book illustration, colorful, whimsical"
# - "a magical forest with talking animals, children's book style, bright colors"
# - "a young explorer discovering a hidden treasure, cartoon style, adventure theme"

# TODO: Generate and display your images


## Summary

In this notebook, you learned:

✅ How diffusion models work (forward and reverse diffusion)  
✅ Text-to-image generation with Stable Diffusion  
✅ Prompt engineering techniques  
✅ Key parameters for controlling generation  
✅ Comparison between VAEs and Diffusion Models  

### Next Steps

- Explore image-to-image generation
- Try inpainting and outpainting
- Experiment with different diffusion models (SDXL, Stable Diffusion 2.1)
- Learn about ControlNet for precise control

### Resources

- [Hugging Face Diffusers Documentation](https://huggingface.co/docs/diffusers)
- [Stable Diffusion Prompt Guide](https://stable-diffusion-art.com/prompt-guide/)
- [Understanding Diffusion Models (Paper)](https://arxiv.org/abs/2208.11970)